# Getting started

The public workflow is:

1. declare fields and gauge groups,
2. write a `Model(...)`,
3. compile with `model.lagrangian()`,
4. extract vertices with `feynman_rule(...)`.

This notebook is the short tour. The other notebooks in this folder take one
subject further:

- `indices.ipynb` — `IndexType`, Spenso slots, custom index families
- `flavor.ipynb` — flavor classes and `flavor_expand`
- `field_strengths.ipynb` — `FS`, $F^3$, $F^4$, mixed groups
- `nested_derivatives.ipynb` — nested `DC`, `PartialD`, and `FS`
- `field_transformations.ipynb` — EWSB, mixing, projectors
- `compiled_operators.ipynb` — operators on a compiled Lagrangian, IBP, Symbolica export
- `gauge_and_brst.ipynb` — infinitesimal gauge variation and BRST


## Setup


In [1]:
import re
import sys
from fractions import Fraction
from pathlib import Path

from symbolica import Expression, S

ROOT = Path.cwd().resolve()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
SRC = ROOT / "src"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

ANSI_ESCAPE_RE = re.compile(r"\x1B\[[0-?]*[ -/]*[@-~]")


def clean(text):
    return ANSI_ESCAPE_RE.sub("", str(text))


def show(title, result):
    print("==========")
    print(title)
    if isinstance(result, dict):
        print(f"{len(result)} vertex signature(s)")
        print()
        for signature, expression in result.items():
            print("Vertex:", signature)
            print("Rule:", clean(expression))
            print()
    else:
        print(clean(result))
        print()


def show_model(model, *fields, compact_form=None, sum_notation=None, simplify=True):
    source_terms = model.lagrangian_decl.source_terms
    if source_terms:
        lagrangian_source = (
            sum(source_terms[1:], source_terms[0])
            if len(source_terms) > 1
            else source_terms[0]
        )
        show("Lagrangian", lagrangian_source)
    lagrangian = model.lagrangian()
    if fields:
        show("Feynman Rule", lagrangian.feynman_rule(*fields, include_delta=False, simplify=simplify))
    else:
        show("Feynman Rules", lagrangian.feynman_rule(include_delta=False, simplify=simplify))
    if compact_form is not None:
        show("Compact Form", compact_form)
    if sum_notation is not None:
        show("Sum Notation", sum_notation)

from feynpy import (
    COLOR_ADJ_INDEX,
    COLOR_FUND_INDEX,
    LORENTZ_INDEX,
    SPINOR_INDEX,
    WEAK_ADJ_INDEX,
    WEAK_FUND_INDEX,
    DC,
    Field,
    FS,
    Gamma,
    GaugeFixing,
    GaugeGroup,
    GaugeRepresentation,
    GhostField,
    GhostLagrangian,
    Metric,
    Model,
    PartialD,
)
from symbolic.spenso_structures import (
    gauge_generator,
    structure_constant,
    weak_gauge_generator,
    weak_structure_constant,
)
from symbolic.vertex_engine import I


## Fields and gauge groups

Use the built-in `IndexType` constants. A field is declared by spin, indices,
and (when needed) charges. The conjugated partner of a non-self-conjugate
field is written `.bar`.


In [2]:
mu, nu = S("mu"), S("nu")
lam4, y, e, gs, g1, g2 = S("lam4"), S("y"), S("e"), S("gs"), S("g1"), S("g2")
Qe, YL, YH, xi = S("Qe"), S("YL"), S("YH"), S("xi")

Phi = Field("Phi", spin=0, self_conjugate=True, symbol=S("phi"))
Psi = Field(
    "Psi",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("psi"),
    conjugate_symbol=S("psibar"),
    indices=(SPINOR_INDEX,),
)
Electron = Field(
    "e",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("e"),
    conjugate_symbol=S("ebar"),
    indices=(SPINOR_INDEX,),
    quantum_numbers={"Q": Qe},
)
Quark = Field(
    "q",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("q"),
    conjugate_symbol=S("qbar"),
    indices=(SPINOR_INDEX, COLOR_FUND_INDEX),
)
Photon = Field("A", spin=1, self_conjugate=True, symbol=S("A"), indices=(LORENTZ_INDEX,))
Gluon = Field(
    "G",
    spin=1,
    self_conjugate=True,
    symbol=S("G"),
    indices=(LORENTZ_INDEX, COLOR_ADJ_INDEX),
)
GhostG = GhostField("ghG", ghost_of=Gluon, indices=(COLOR_ADJ_INDEX,))
W = Field(
    "W",
    spin=1,
    self_conjugate=True,
    symbol=S("W"),
    indices=(LORENTZ_INDEX, WEAK_ADJ_INDEX),
)
Higgs = Field(
    "H",
    spin=0,
    self_conjugate=False,
    symbol=S("H"),
    conjugate_symbol=S("Hdag"),
    indices=(WEAK_FUND_INDEX,),
    quantum_numbers={"Y": YH},
)
LDoublet = Field(
    "L",
    spin=Fraction(1, 2),
    self_conjugate=False,
    symbol=S("L"),
    conjugate_symbol=S("Lbar"),
    indices=(SPINOR_INDEX, WEAK_FUND_INDEX),
    quantum_numbers={"Y": YL},
)

COLOR_FUND_REP = GaugeRepresentation(
    index=COLOR_FUND_INDEX,
    generator_builder=gauge_generator,
    name="fundamental",
)
WEAK_DOUBLET_REP = GaugeRepresentation(
    index=WEAK_FUND_INDEX,
    generator_builder=weak_gauge_generator,
    name="doublet",
)

U1QED = GaugeGroup(name="U1QED", abelian=True, coupling=e, gauge_boson=Photon, charge="Q")
SU3C = GaugeGroup(
    name="SU3C",
    abelian=False,
    coupling=gs,
    gauge_boson=Gluon,
    ghost_field=GhostG,
    structure_constant=structure_constant,
    representations=(COLOR_FUND_REP,),
)
SU2L = GaugeGroup(
    name="SU2L",
    abelian=False,
    coupling=g2,
    gauge_boson=W,
    structure_constant=weak_structure_constant,
    representations=(WEAK_DOUBLET_REP,),
)


## Local operators

A `Model` accepts ordinary Python products of fields. `show_model` prints the
declared Lagrangian and the extracted Feynman rules.


In [3]:
scalar_model = Model(lam4 * Phi * Phi * Phi * Phi)
show_model(scalar_model, Phi, Phi, Phi, Phi)

yukawa_model = Model(y * Psi.bar * Psi * Phi)
show_model(yukawa_model, Psi.bar, Psi, Phi)


Lagrangian
lam4 * Phi * Phi * Phi * Phi

Feynman Rule
24𝑖*lam4

Lagrangian
y * Psi.bar * Psi * Phi

Feynman Rule
1𝑖*y*g(bis(4, i1),bis(4, i2))



Nested `PartialD` is allowed. Here a local $\phi^4$ term and a
$\phi^3\partial^2\phi$ term contribute to the same four-scalar vertex.


In [4]:
gBox = S("gBox")
derivative_model = Model(
    lam4 * Phi * Phi * Phi * Phi
    + gBox * Phi * Phi * Phi * PartialD(PartialD(Phi, mu), nu) * Metric(mu, nu)
)
show_model(derivative_model, Phi, Phi, Phi, Phi)


Lagrangian
lam4 * Phi * Phi * Phi * Phi + gBox * Phi * Phi * Phi * PartialD(Phi, mu, nu) * Metric(mu, nu)

Feynman Rule
24𝑖*lam4-6𝑖*gBox*pcomp(q1,mu1_int)^2-6𝑖*gBox*pcomp(q2,mu1_int)^2-6𝑖*gBox*pcomp(q3,mu1_int)^2-6𝑖*gBox*pcomp(q4,mu1_int)^2



## Gauge currents

`DC(...)` expands using the charges and representations declared on the field.


In [5]:
qed_model = Model(
    I * Electron.bar * Gamma(mu) * DC(Electron, mu),
    gauge_groups=(U1QED,),
)
show_model(qed_model, Electron.bar, Electron, Photon)

qcd_model = Model(
    I * Quark.bar * Gamma(mu) * DC(Quark, mu),
    gauge_groups=(SU3C,),
)
show_model(qcd_model, Quark.bar, Quark, Gluon)


Lagrangian
1𝑖 * e.bar * Gamma(mu) * DC(e, mu)

Feynman Rule
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

Lagrangian
1𝑖 * q.bar * Gamma(mu) * DC(q, mu)

Feynman Rule
1𝑖*gs*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(8, a3),cof(3, c1),cof(3, c2))



## Yang–Mills, gauge fixing, and ghosts

`FS(...)` generates the pure-gauge sector. `GaugeFixing` and `GhostLagrangian`
are the ordinary linear covariant gauge and Faddeev–Popov terms.


In [6]:
ym_model = Model(
    -(Expression.num(1) / Expression.num(4)) * FS(SU3C, mu, nu, S("aC")) * FS(SU3C, mu, nu, S("aC")),
    gauge_groups=(SU3C,),
)
show_model(ym_model)

gf_model = Model(GaugeFixing(SU3C, xi=xi), gauge_groups=(SU3C,))
show_model(gf_model)

ghost_model = Model(GhostLagrangian(SU3C), gauge_groups=(SU3C,))
show_model(ghost_model, GhostG.bar, Gluon, GhostG)


Lagrangian
-1/4 * FS(SU3C, mu, nu, aC) * FS(SU3C, mu, nu, aC)

Feynman Rules
3 vertex signature(s)

Vertex: ('G', 'G')
Rule: 1𝑖*g(mink(4, mu1),mink(4, mu2))*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)-1𝑖*g(coad(8, a1),coad(8, a2))*pcomp(q1,mu2)*pcomp(q2,mu1)

Vertex: ('G', 'G', 'G')
Rule: -gs*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu2)+gs*g(mink(4, mu3),mink(4, mu1))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu2)+gs*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu1)-gs*g(mink(4, mu3),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q3,mu1)+gs*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q1,mu3)-gs*g(mink(4, mu1),mink(4, mu2))*f(coad(8, a1),coad(8, a2),coad(8, a3))*pcomp(q2,mu3)

Vertex: ('G', 'G', 'G', 'G')
Rule: -1𝑖*gs^2*g(mink(4, mu3),mink(4, mu1))*g(mink(4, mu2),mink(4, mu4))*f(coad(8, a1),coad(8, a2),coad(8, aC))*f(coad(8, a3),coad(8, a4),coad(8, aC))

## A small electroweak slice

One left-handed doublet and one Higgs doublet under $\mathrm{SU}(2)_L$ already
produce currents and the scalar-gauge contact term from $(D_\mu H)^\dagger(D^\mu H)$.


In [7]:
ew_model = Model(
    I * LDoublet.bar * Gamma(mu) * DC(LDoublet, mu)
    + DC(Higgs.bar, mu) * DC(Higgs, mu),
    gauge_groups=(SU2L,),
)
show_model(ew_model)


Lagrangian
1𝑖 * L.bar * Gamma(mu) * DC(L, mu) + DC(H.bar, mu) * DC(H, mu)

Feynman Rules
5 vertex signature(s)

Vertex: ('L.bar', 'L', 'W')
Rule: 1𝑖*g2*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*t(coad(3, aw3),cof(2, w1),cof(2, w2))

Vertex: ('L.bar', 'L')
Rule: 1𝑖*g(cof(2, w1),cof(2, w2))*gamma(bis(4, i1),bis(4, i2),mink(4, mu1_int))*pcomp(q2,mu1_int)

Vertex: ('H.bar', 'H', 'W')
Rule: -1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q1,mu3)+1𝑖*g2*t(coad(3, aw3),cof(2, w1),cof(2, w2))*pcomp(q2,mu3)

Vertex: ('H.bar', 'H', 'W', 'W')
Rule: 1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w_mid_H_SU2L),cof(2, w2))*t(coad(3, aw4),cof(2, w1),cof(2, w_mid_H_SU2L))+1𝑖*g2^2*g(mink(4, mu3),mink(4, mu4))*t(coad(3, aw3),cof(2, w1),cof(2, w_mid_H_SU2L))*t(coad(3, aw4),cof(2, w_mid_H_SU2L),cof(2, w2))

Vertex: ('H.bar', 'H')
Rule: -1𝑖*g(cof(2, w1),cof(2, w2))*pcomp(q1,mu1_int)*pcomp(q2,mu1_int)



## Vertex extraction

- pass explicit fields to get one rule,
- pass no fields to enumerate every signature,
- `include_delta=True` keeps the overall momentum-conserving factor.

`vertex_signatures(...)` inspects the compiled content without extracting every
expression.


In [8]:
qed_L = qed_model.lagrangian()

show(
    "QED vertex without Delta",
    qed_L.feynman_rule(Electron.bar, Electron, Photon, include_delta=False),
)
show(
    "QED vertex with labelled momenta",
    qed_L.feynman_rule(
        Electron.bar,
        Electron,
        Photon,
        momenta=[S("p_in"), S("p_out"), S("k")],
        include_delta=False,
    ),
)

show("QCD signatures", [s.names for s in qcd_model.lagrangian().vertex_signatures()])
show("EW signatures containing W", [s.names for s in ew_model.lagrangian().vertex_signatures(contains_fields=(W,))])


QED vertex without Delta
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

QED vertex with labelled momenta
1𝑖*e*Qe*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))

QCD signatures
[('q.bar', 'q'), ('q.bar', 'q', 'G')]

EW signatures containing W
[('H.bar', 'H', 'W'), ('L.bar', 'L', 'W'), ('H.bar', 'H', 'W', 'W')]



## Validation, reports, and output controls

The same compiled object can also be inspected before asking for individual
rules. `validate()` returns structured diagnostics, `vertex_report(...)` groups
available signatures, and zero-argument `feynman_rule(...)` can use either
readable names or field objects as keys.


In [9]:
qed_validation = qed_model.validate()
qed_compiled_validation = qed_L.validate()
show(
    "validation summaries",
    (
        f"model ok={qed_validation.ok}, "
        f"model errors={len(qed_validation.errors)}, "
        f"compiled ok={qed_compiled_validation.ok}, "
        f"compiled errors={len(qed_compiled_validation.errors)}"
    ),
)

for title, report in (
    ("QCD matter signatures containing G", qcd_model.lagrangian().vertex_report(contains_fields=(Gluon,))),
    ("Yang-Mills pure-gauge signatures", ym_model.lagrangian().vertex_report(sector="pure_gauge")),
    ("Ghost-sector signatures", ghost_model.lagrangian().vertex_report(sector="ghost")),
):
    show(
        title,
        [
            {
                "names": signature.names,
                "terms": signature.term_count,
                "sectors": signature.sectors,
            }
            for signature in report.signatures
        ],
    )

def field_key_name(field_arg):
    if hasattr(field_arg, "field"):
        return f"{field_arg.field.name}.bar"
    return field_arg.name


qed_keys_as_fields = qed_L.feynman_rule(arity=3, key_format="fields")
show(
    "zero-argument keys as fields",
    [tuple(field_key_name(field) for field in fields) for fields in qed_keys_as_fields],
)
show(
    "QED vertex with external wavefunctions and delta",
    qed_L.feynman_rule(
        Electron.bar,
        Electron,
        Photon,
        include_delta=True,
        strip_externals=False,
    ),
)


validation summaries
model ok=True, model errors=0, compiled ok=True, compiled errors=0

QCD matter signatures containing G
[{'names': ('q.bar', 'q', 'G'), 'terms': 1, 'sectors': ('matter',)}]

Yang-Mills pure-gauge signatures
[{'names': ('G', 'G'), 'terms': 4, 'sectors': ('pure_gauge',)}, {'names': ('G', 'G', 'G'), 'terms': 4, 'sectors': ('pure_gauge',)}, {'names': ('G', 'G', 'G', 'G'), 'terms': 1, 'sectors': ('pure_gauge',)}]

Ghost-sector signatures
[{'names': ('ghG.bar', 'ghG'), 'terms': 1, 'sectors': ('ghost',)}, {'names': ('ghG.bar', 'G', 'ghG'), 'terms': 1, 'sectors': ('ghost',)}]

zero-argument keys as fields
[('e.bar', 'e', 'A')]

QED vertex with external wavefunctions and delta
1𝑖*e*Qe*(2*𝜋)^d*gamma(bis(4, i1),bis(4, i2),mink(4, mu3))*U(A,q3)*Delta(q1+q2+q3)



## Conventions

- derivatives map to $-i p_\mu$,
- public `feynman_rule(...)` supplies the overall $+i$,
- matter uses $D_\mu = \partial_\mu - i g A_\mu$,
- the momentum-conservation delta is omitted unless `include_delta=True`.

The Standard Model and SMEFT implementations live under `models/SM` and
`models/SMEFT`.
